# Block 3 — Auto Loader: incremental file ingestion into Bronze

Ingests `landing/orders/` into `retail.bronze.orders_stream` with Auto Loader.
Proves four things: incremental file discovery, idempotent re-runs, schema evolution
on a new column, and checkpoint recovery after a mid-stream failure.

**Scope, stated honestly:** there is no broker and no live producer. "Arrival" is a file
appearing in a container. Auto Loader's job here is *file discovery*, not stream transport.

In [0]:
STG        = "stgaccdeprep"
LANDING    = f"abfss://landing@{STG}.dfs.core.windows.net"

SRC        = f"{LANDING}/orders"
SCHEMA_LOC = f"{LANDING}/_control/orders_stream/schema"
CHKPT_LOC  = f"{LANDING}/_control/orders_stream/checkpoint"
TARGET     = "retail.bronze.orders_stream"

from pyspark.sql import functions as F

display(dbutils.fs.ls(SRC))

In [0]:
display(spark.read.text(f"{SRC}/orders_batch_01.csv").limit(3))

## Schema hints, not a declared schema

Block 1 declared the orders schema explicitly — correct for a batch read of a known file.
Here it would be wrong: a fully declared schema sets `schemaEvolutionMode` to `none`, and
the whole point of this block is to watch a new column arrive.

Hints give typed columns where the contract is known while leaving the stream open to
new ones. The cost is that a missing column no longer fails at read time — it fails later,
in whatever consumes it.

In [0]:
HINTS = """
  order_id string,
  customer_id string,
  order_ts timestamp,
  updated_ts timestamp,
  amount decimal(12,2),
  status string,
  payment_method string
"""

raw = (spark.readStream.format("cloudFiles")
       .option("cloudFiles.format", "csv")
       .option("cloudFiles.schemaLocation", SCHEMA_LOC)
       .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
       .option("cloudFiles.inferColumnTypes", "true")
       .option("cloudFiles.schemaHints", HINTS)
       .option("header", "true")
       .option("rescuedDataColumn", "_rescued_data")
       .load(SRC))

bronze_stream = (raw
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingest_ts",  F.current_timestamp()))

bronze_stream.printSchema()

## Two locations, and they are not the same thing

`cloudFiles.schemaLocation` stores the inferred schema and its evolution history.
`checkpointLocation` stores stream progress — which files have been consumed.

Point them at the same path and the stream appears to work until the first restart.
This is the classic first-run misconfiguration.

In [0]:
q = (bronze_stream.writeStream
     .format("delta")
     .option("checkpointLocation", CHKPT_LOC)
     .option("mergeSchema", "true")
     .trigger(availableNow=True)
     .toTable(TARGET))

q.awaitTermination()

for p in q.recentProgress:
    print(f"batch {p['batchId']}: {p['numInputRows']} rows")

In [0]:
display(spark.sql(f"""
  SELECT _source_file, count(*) AS rows, min(_ingest_ts) AS ingested_at
  FROM {TARGET}
  GROUP BY _source_file
  ORDER BY _source_file
"""))

print("total rows:", spark.table(TARGET).count())
print("rescued rows:", spark.table(TARGET).filter("_rescued_data IS NOT NULL").count())

## Run 2 — no new files

Re-run cell 7 with nothing new in `landing/orders/`.
The checkpoint says every file is consumed, so the stream finds no work.
Result: batch 1: 0 rows, table unchanged at 640.

## Run 3 — a file arrives

`orders_batch_02.csv` uploaded to `landing/orders/`. Seven columns, same contract as batch 01.
Auto Loader discovers it by listing the directory and comparing against the checkpoint.

Result: batch 1 → 340 rows — only the new file. Table 640 → 980.
Batch ID 1 again: the empty run planned no batch, so the counter never advanced.

## Run 4 — schema evolution, and a real failure

`orders_batch_03.csv` adds a `channel` column (WEB / STORE / PARTNER / MOBILE_APP).
With `schemaEvolutionMode = addNewColumns`, Auto Loader records the new schema and then
**deliberately fails the stream**. Restarting picks up the updated schema and continues.

Failing rather than silently absorbing is the right default: a new column is a contract
change, and something downstream may need to know before rows carrying it land.

Result: fails with UNKNOWN_FIELD_EXCEPTION naming channel, then succeeds on restart with 240 rows. Table 980 → 1220, batch_01 still 640

In [0]:
display(spark.sql(f"""
  SELECT _source_file, count(*) AS rows, min(_ingest_ts) AS ingested_at
  FROM {TARGET}
  GROUP BY _source_file
  ORDER BY _source_file
"""))

print("total rows:", spark.table(TARGET).count())
print("rescued rows:", spark.table(TARGET).filter("_rescued_data IS NOT NULL").count())

display(spark.sql(f"""
  SELECT channel, count(*) AS rows
  FROM {TARGET}
  GROUP BY channel
  ORDER BY rows DESC
"""))

## The control state on disk

Two directories, two jobs. The checkpoint holds `offsets/`, `commits/`, `metadata` and
`sources/`; the schema location holds only `_schemas/`. They share nothing.

Three commits from five runs of the write cell. The empty run planned no batch at all;
the failed run planned one and never committed it. Commits record work completed, not
attempts made — which is what makes the failed batch replayable rather than lost.

In [0]:
print("checkpoint:")
display(dbutils.fs.ls(CHKPT_LOC))

print("schema location:")
display(dbutils.fs.ls(SCHEMA_LOC))

In [0]:
print("schema versions:")
display(dbutils.fs.ls(f"{SCHEMA_LOC}/_schemas"))

print("commits:")
display(dbutils.fs.ls(f"{CHKPT_LOC}/commits"))

print("offsets:")
display(dbutils.fs.ls(f"{CHKPT_LOC}/offsets"))

## Exactly-once, as two files and a frozen clock

`offsets/2` was written by the run that failed; `commits/2` by the run that succeeded.
Offset ahead of processing, commit after — a batch with an offset and no commit is replayed.

`batchTimestampMs` is stored in that offset, so the replayed rows carry `_ingest_ts` from
the failed run's clock, not the successful one's. Replay reproduces the batch exactly,
including its notion of now.

In [0]:
print(dbutils.fs.head(f"{CHKPT_LOC}/offsets/2"))